# BSI Examples

This notebook shows three small examples:

1. original multi-armed bandit BSI with Gaussian rewards;
2. contextual BSI using the `contextual/` core API;
3. sub-Gaussian multi-armed bandit BSI using Beta rewards.

The settings are intentionally small so the notebook can run quickly. Increase the horizons and Monte Carlo repetitions for serious experiments.

In [ ]:
from pathlib import Path
import argparse
import json
import subprocess
import sys

import numpy as np

ROOT = Path.cwd()
RESULTS = ROOT / "example_results"
RESULTS.mkdir(exist_ok=True)

print(f"Working directory: {ROOT}")

## 1. Original MAB BSI: Gaussian Rewards

This example uses the original root-level `run.py`. The logging policy is epsilon-greedy and the evaluation policy is Gaussian Thompson sampling.

In [ ]:
mab_cmd = [
    sys.executable, "run.py",
    "--env", "normal",
    "--mus", "0.1", "0.2", "1.0",
    "--sigmas", "1.0", "1.0", "1.0",
    "--pi0", "epsilon_greedy",
    "--pi1", "ts_normal",
    "--T", "50",
    "--T_offline", "100",
    "--offline_reps", "20",
    "--infer_reps", "200",
    "--no-select_M",
    "--n_policy_value_mc", "500",
    "--n_eval_prob_mc", "200",
    "--alphas", "0.10",
    "--no-save_logged_data",
    "--save_dir", str(RESULTS / "mab_gaussian"),
    "--tag", "quick_mab_gaussian",
]

print(" ".join(mab_cmd))
subprocess.run(mab_cmd, check=True)

## 2. Contextual BSI: Linear Gaussian Contextual Bandit

The contextual extension is intentionally kept under `contextual/`. This example uses the core API directly: simulate offline data, fit the contextual reward model, then run BSI for the target contextual policy.

In [ ]:
from contextual.contextual_bsi import ContextualParametricBSI
from contextual.simulation import (
    collect_offline_data,
    context_sampler,
    default_params,
    make_policy_builder,
    make_reward_model,
)

ctx_args = argparse.Namespace(
    env="linear_gaussian",
    n_actions=3,
    context_dim=2,
    T=50,
    T_offline=100,
    pi0="uniform",
    pi1="contextual_epsilon",
    epsilon0=0.1,
    epsilon1=0.1,
    obs_sigma=1.0,
    reward_sigma=1.0,
    ts_prob_mc=200,
    param_scenario="default",
    context_var=1.0,
    policy_explore_untried=False,
    seed=2026,
)

true_params = default_params(
    ctx_args.env,
    ctx_args.n_actions,
    ctx_args.context_dim,
    ctx_args.seed,
    ctx_args.param_scenario,
)
offline = collect_offline_data(ctx_args, true_params, rep_idx=0)
reward_model = make_reward_model(ctx_args, adaptive_behavior=False)

bsi = ContextualParametricBSI(
    reward_model=reward_model,
    eval_policy_builder=make_policy_builder(ctx_args, ctx_args.pi1, ctx_args.epsilon1, true_params),
    context_sampler=context_sampler(ctx_args.context_dim, ctx_args.context_var),
    T=ctx_args.T,
    algo_seed=ctx_args.seed + 1,
    context_seed=ctx_args.seed + 2,
)

ctx_result = bsi.run(offline, alphas=[0.10], n_reps=500)
ctx_interval = (
    ctx_result.center - ctx_result.ci_width[0.10],
    ctx_result.center + ctx_result.ci_width[0.10],
)

print("theta_hat:", ctx_result.center)
print("90% BSI CI:", ctx_interval)
print("lambda_hat shape:", ctx_result.lambda_hat.shape)
print("gradient norm:", np.linalg.norm(ctx_result.gradient))

## 3. Sub-Gaussian MAB BSI: Beta Rewards

This example uses the original MAB runner with `env=beta`. The Hoeffding variance rule treats bounded Beta rewards as sub-Gaussian with fixed standard deviation `0.5`.

In [ ]:
beta_cmd = [
    sys.executable, "run.py",
    "--env", "beta",
    "--beta_alphas", "0.35", "0.5", "0.5",
    "--beta_betas", "0.65", "0.5", "0.5",
    "--var_estimation_beta", "hoeffding",
    "--pi0", "ts_normal",
    "--pi1", "ts_normal",
    "--T", "50",
    "--T_offline", "100",
    "--offline_reps", "20",
    "--infer_reps", "200",
    "--no-select_M",
    "--n_policy_value_mc", "500",
    "--n_eval_prob_mc", "200",
    "--alphas", "0.10",
    "--no-save_logged_data",
    "--save_dir", str(RESULTS / "mab_beta_subgaussian"),
    "--tag", "quick_mab_beta",
]

print(" ".join(beta_cmd))
subprocess.run(beta_cmd, check=True)

## Optional: Select the Contextual Inner Monte Carlo Size `M`

`contextual.select_inner_reps` implements the pilot-bootstrap criterion for choosing the number of inner Monte Carlo replications used by contextual BSI.

In [ ]:
select_m_cmd = [
    sys.executable, "-m", "contextual.select_inner_reps",
    "--envs", "linear_gaussian",
    "--pairs", "uni_ts",
    "--T", "50",
    "--T_offline_grid", "100",
    "--m0", "100",
    "--B", "100",
    "--out", str(RESULTS / "selected_inner_reps.csv"),
]

print(" ".join(select_m_cmd))
subprocess.run(select_m_cmd, check=True)